# AI Productivity Toolkit — Prompt Engineering & AI Productivity

**Objective:** Practice zero-shot and few-shot prompting, roles/context/instructions/constraints, structured JSON output, summarization, classification, generation, AI-assisted coding/debugging, and prompt evaluation.

**How to use this notebook:** Run each cell. The notebook builds reusable prompts; copy them into an AI assistant to test the actual model response. Record the real outputs and scores in `reports/prompt_evaluation_report.md`. Example snippets in this notebook are demonstrations, not claims that an AI model was called here.


## 1. Prompt anatomy
A useful prompt specifies a role, context, task, constraints, and output format. Few-shot prompts also include examples.

In [ ]:
def build_prompt(role, context, task, constraints, output_format):
    constraint_lines = "\n".join(f"- {item}" for item in constraints) or "- No extra constraints provided."
    return (
        f"Role: {role}\n"
        f"Context: {context}\n"
        f"Task: {task}\n"
        f"Constraints:\n{constraint_lines}\n"
        f"Output format: {output_format}"
    )

example = build_prompt(
    role="You are a patient programming tutor.",
    context="The learner is new to Java loops.",
    task="Explain a for-loop using a small example.",
    constraints=["Use simple language", "Keep it under 150 words", "Do not skip the stopping condition"],
    output_format="Short explanation, code snippet, and one practice question"
)
print(example)

## 2. Zero-shot vs. few-shot classification
Zero-shot gives instructions without examples. Few-shot adds labeled examples to show the expected pattern.

In [ ]:
def classification_prompt(message, few_shot=False):
    prompt = (
        "Classify the message as exactly one of: positive, neutral, negative.\n"
        "Return only valid JSON with keys \"label\" and \"reason\". Do not include Markdown.\n"
    )
    if few_shot:
        prompt += (            'Examples:\n'
            'Message: "I love the helpful support."\n'
            'Output: {"label":"positive","reason":"The message expresses satisfaction."}\n'
            'Message: "The meeting starts at 3 PM."\n'
            'Output: {"label":"neutral","reason":"The message is factual without sentiment."}\n'
            'Message: "The app crashes every time."\n'
            'Output: {"label":"negative","reason":"The message reports a problem."}\n'
        )
    return prompt + f"Message: {message}"

sample_message = "The app is easy to use, but it keeps crashing."
print("ZERO-SHOT PROMPT\n")
print(classification_prompt(sample_message))
print("\nFEW-SHOT PROMPT\n")
print(classification_prompt(sample_message, few_shot=True))

**Practice:** Paste both prompts into the same AI assistant with the same input. Compare label consistency, reasons, and whether the output is valid JSON. Record what actually happened rather than assuming few-shot always wins.

## 3. Structured output (JSON)
When a result will be consumed by code, specify exact keys and what to do when information is missing.

In [ ]:
def json_extraction_prompt(text):
    return build_prompt(
        role="You extract fields precisely.",
        context="The output will be parsed by a program.",
        task="Extract a person's name, date, task, and priority from the text.",
        constraints=[
            "Return valid JSON only, without Markdown fences.",
            "Use keys person, date, task, priority.",
            "Use null for missing details; do not guess.",
            "Format dates as YYYY-MM-DD when possible."
        ],
        output_format='{"person": string|null, "date": string|null, "task": string, "priority": "low"|"medium"|"high"|null}'
    ) + f"\nSource text: {text}"

print(json_extraction_prompt("Priya will submit the project report on 2026-10-20. Priority is high."))

In [ ]:
# Example of validating JSON returned by an AI assistant.
# Replace sample_response with the actual model response you want to inspect.
import json

sample_response = '{"person":"Priya","date":"2026-10-20","task":"submit the project report","priority":"high"}'
try:
    parsed = json.loads(sample_response)
    required_keys = {"person", "date", "task", "priority"}
    assert set(parsed.keys()) == required_keys, "Unexpected JSON keys"
    print("Valid JSON with expected keys:", parsed)
except (json.JSONDecodeError, AssertionError) as error:
    print("Response needs correction:", error)

## 4. Summarization workflow

In [ ]:
def summarization_prompt(text, audience="a busy student", bullet_count=5):
    return build_prompt(
        role="You are a careful summarization assistant.",
        context=f"The audience is {audience}.",
        task=f"Summarize the source in {bullet_count} bullet points.",
        constraints=[
            "Preserve the main claims and important details.",
            "Do not add facts absent from the source.",
            "Mention uncertainty if the source is unclear."
        ],
        output_format="Summary bullets, followed by action items if any"
    ) + f"\nSource text:\n{text}"

source_text = """A college coding club held a workshop on Saturday. Forty students attended. The session covered Git basics, branching, and pull requests. Organizers plan to share slides next week, but the next workshop date has not been decided."""
print(summarization_prompt(source_text, bullet_count=3))

## 5. Generation workflows: email and resume

In [ ]:
email_prompt = build_prompt(
    role="You are a professional communication assistant.",
    context="A student is writing to a faculty member about an assignment.",
    task="Draft a polite email requesting clarification about the submission format.",
    constraints=["Maximum 120 words", "Use a natural tone", "Do not invent deadlines"],
    output_format="Subject line plus email body"
)
resume_prompt = build_prompt(
    role="You are a resume-writing assistant.",
    context="A student has built a small Java JDBC CRUD application.",
    task="Turn the supplied notes into one honest resume bullet.",
    constraints=["Do not invent metrics", "Mention only tools actually used", "Start with an action verb"],
    output_format="One resume bullet"
)
print("EMAIL PROMPT\n", email_prompt)
print("\nRESUME PROMPT\n", resume_prompt)

## 6. AI-assisted coding and debugging

In [ ]:
debug_prompt = build_prompt(
    role="You are a patient Java debugging assistant.",
    context="A beginner uses IntelliJ IDEA and Maven for a JDBC project.",
    task="Diagnose the error: No suitable driver found for jdbc:mysql://localhost:3306/studentdb.",
    constraints=[
        "Do not rewrite unrelated code.",
        "Check whether the MySQL Connector/J dependency is included and loaded.",
        "Separate likely causes from confirmed facts.",
        "Give verification steps after the fix."
    ],
    output_format="Likely cause, ordered checks, minimal fix, verification"
)
print(debug_prompt)

**Coding safety check:** Never paste passwords, API keys, or private user data into a prompt. Run generated code, test normal and edge cases, and inspect changes before using them.

## 7. Research assistance

In [ ]:
research_prompt = build_prompt(
    role="You are a careful research assistant.",
    context="The result will be used for a college assignment.",
    task="Create a research plan for AI-assisted learning tools.",
    constraints=[
        "Prioritize primary and authoritative sources.",
        "Do not invent references, quotations, URLs, or publication dates.",
        "Separate facts, disputed points, and assumptions.",
        "List claims that still need verification."
    ],
    output_format="Search terms, source plan, outline, verification checklist, open questions"
)
print(research_prompt)

## 8. Prompt evaluation and testing report

Use a repeatable test plan instead of rating a prompt from one response. Compare a baseline prompt with an improved prompt on the same inputs, using the same AI tool/model where possible.

In [ ]:
criteria = {
    "clarity": "Is the task unambiguous?",
    "context": "Is relevant background provided?",
    "constraints": "Are limits and exclusions explicit?",
    "output_format": "Can the response structure be checked?",
    "reliability": "Does the prompt discourage guessing?",
    "usefulness": "Does the response meet the user's need?",
}

print("Score each criterion from 1 (weak) to 5 (excellent).")
for criterion, question in criteria.items():
    print(f"- {criterion}: {question}")

print("\nTest report template:")
print("Test | Baseline prompt | Improved feature | Actual result | Score / 30 | Next change")

## 9. Reflection — complete before submission
Write your own brief answers after trying the prompts:

1. Which prompt improved most after adding context or constraints?
2. Did few-shot prompting improve consistency for your test input? Show evidence from the response.
3. Did the model return valid JSON? What validation did you run?
4. What is one risk of using AI-generated code or research?
5. What prompt would you improve next, and why?

**Submission reminder:** Push the notebook and supporting files to GitHub, save/upload this notebook in Google Colab, and fill the report with actual test results before publishing your LinkedIn post.